# Klasifikasi DemogPairs Menggunakan ViT (Wajah dan Umur) & Gaussian Naive Bayes

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

joblib.parallel_backend('threading')

## Load Dataset

In [2]:
data = u.load_demogpairs()
pd.DataFrame(data)

,db_code,image_path,full_path,label,label_idx
0,CWF,able_wanamakok/002.jpg,dataset/demogpairs/images\able_wanamakok/002.jpg,Asian_Females,5
1,CWF,able_wanamakok/004.jpg,dataset/demogpairs/images\able_wanamakok/004.jpg,Asian_Females,5
2,CWF,able_wanamakok/007.jpg,dataset/demogpairs/images\able_wanamakok/007.jpg,Asian_Females,5
3,CWF,able_wanamakok/008.jpg,dataset/demogpairs/images\able_wanamakok/008.jpg,Asian_Females,5
4,CWF,able_wanamakok/012.jpg,dataset/demogpairs/images\able_wanamakok/012.jpg,Asian_Females,5
...,...,...,...,...,...
10795,CWF,zachary_quinto/177.jpg,dataset/demogpairs/images\zachary_quinto/177.jpg,White_Males,3
10796,CWF,zachary_quinto/214.jpg,dataset/demogpairs/images\zachary_quinto/214.jpg,White_Males,3
10797,CWF,zachary_quinto/217.jpg,dataset/demogpairs/images\zachary_quinto/217.jpg,White_Males,3
10798,CWF,zachary_quinto/218.jpg,dataset/demogpairs/images\zachary_quinto/218.jpg,White_Males,3


## Load Fitur

In [3]:
face_features = joblib.load('features/demogpairs_vit-face.pkl')
age_features = joblib.load('features/demogpairs_vit-age.pkl')
features = {}
for d in tqdm(data):
    key = d['image_path']
    features[key] = np.array(list(face_features[key]) + list(age_features[key]))
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

100%|█████████████████████████████████████████████████████████████████████████| 10800/10800 [00:01<00:00, 10052.77it/s]

Jumlah fitur per gambar: 1536


## Split Data

In [4]:
X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
print((len(X_train), len(X_test)))

(8640, 2160)


## Kombinasi Parameter

In [5]:
var_smoothing_values = np.logspace(-9, 2, 40)  # dari 1e-9 sampai 1e2, 40 nilai

grid_params = [
    {
        'scaler': [None, MinMaxScaler()],
        'pca': [None, PCA(n_components=0.5), PCA(n_components=0.75)],
        
        'classifier': [GaussianNB()],
        'classifier__var_smoothing': var_smoothing_values
    },
]

pipeline = Pipeline(steps=[
    ('scaler', None),
    ('pca', None),
    ('classifier', None)
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro',

}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.6),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

GaussianNB: 240 kombinasi


## Klasifikasi

In [6]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models,
    X_train, y_train,
    X_test, y_test,
    target_names=u.demogpairs_classes,
    model_prefix='models/clf_demogpairs_gnb_vit-face-age_',
    results_path='results/demogpairs_gnb_vit-face-age_'
)

sorted_results = pd.DataFrame(evaluation_results).sort_values(by='test_accuracy', ascending=False).to_dict('records')
u.html_br()
_dtable = u.display_table(sorted_results)

Evaluating: GaussianNB


{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.011253355826007646), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}


Accuracy  : 0.8314814814814815
Precision : 0.8343036484691989
Recall    : 0.8314814814814815
F1 Score  : 0.8316522462782968
               precision    recall  f1-score   support

Asian_Females     0.8177    0.8972    0.8556       360
  Asian_Males     0.9112    0.8556    0.8825       360
Black_Females     0.7855    0.7833    0.7844       360
  Black_Males     0.8966    0.8667    0.8814       360
White_Females     0.8385    0.7500    0.7918       360
  White_Males     0.7563    0.8361    0.7942       360

     accuracy                         0.8315      2160
    macro avg     0.8343    0.8315    0.8317      2160
 weighted avg     0.8343    0.8315    0.8317      2160



Class,OvR Accuracy,Precision,Recall,F1-Score,Support
Asian_Females,0.9495370370370371,0.8177215189873418,0.8972222222222223,0.8556291390728478,360
Asian_Males,0.962037037037037,0.9112426035502958,0.8555555555555555,0.8825214899713467,360
Black_Females,0.9282407407407407,0.7855153203342619,0.7833333333333333,0.7844228094575799,360
Black_Males,0.9611111111111111,0.896551724137931,0.8666666666666667,0.8813559322033899,360
White_Females,0.9342592592592592,0.8385093167701864,0.75,0.7917888563049854,360
White_Males,0.9277777777777778,0.7562814070351759,0.8361111111111111,0.7941952506596306,360


Confusion matrix saved: images\cm_gnb_vit-face-age_GaussianNB.png



Confusion Matrix:
                         Asian_Females       Asian_Males     Black_Females       Black_Males     White_Females       White_Males
       Asian_Females               323                 0                12                17                 8                 0
         Asian_Males                 2               308                 4                 1                19                26
       Black_Females                18                 0               282                14                 1                45
         Black_Males                11                 6                27               312                 2                 2
       White_Females                40                14                11                 1               270                24
         White_Males                 1                10                23                 3                22               301


model_name,model_file_path,best_parameters,test_accuracy,test_f1,test_precision,test_recall,parameter_combinations
GaussianNB,models/clf_demogpairs_gnb_vit-face-age_GaussianNB.pkl,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.011253355826007646), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8314814814814815,0.8316522462782968,0.8343036484691989,0.8314814814814815,240


In [7]:
model, training_time = u.load_object('models/clf_demogpairs_gnb_vit-face-age_GaussianNB.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))

{'input_seconds': 1734.0,
 'days': 0,
 'hours': 0,
 'minutes': 28,
 'seconds': 54.0,
 'text': '0 hari 0 jam 28 menit 54.0 detik'}

In [8]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))

{'input_seconds': 10684.0,
 'days': 0,
 'hours': 2,
 'minutes': 58,
 'seconds': 4.0,
 'text': '0 hari 2 jam 58 menit 4.0 detik'}

In [9]:
_dtable = u.display_table(fold_results, n_items=[4, 4], column_widths=['5%', '45%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%'])

No,Params,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Accuracy Mean,F1 Score Mean,Precision Mean,Recall Mean,Train Time Mean
1,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.011253355826007646), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8519,0.8328,0.8414,0.8432,0.8478,0.8434,0.8432,0.8466,0.8434,9.2913
2,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.0058780160722749115), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8501,0.8351,0.842,0.8449,0.8438,0.8432,0.8428,0.8445,0.8432,10.4237
3,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.0058780160722749115), 'pca': 'PCA', 'scaler': None}",0.8478,0.8328,0.8409,0.8426,0.8478,0.8424,0.842,0.8442,0.8424,11.5458
4,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.011253355826007646), 'pca': 'PCA', 'scaler': None}",0.853,0.8333,0.8403,0.8368,0.8472,0.8421,0.8421,0.8456,0.8421,16.8326
...,...,...,...,...,...,...,...,...,...,...,...
237,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(14.251026703029963), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8084,0.8061,0.7922,0.7992,0.7865,0.7985,0.7964,0.8076,0.7985,12.3843
238,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(27.283333764867695), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8073,0.8061,0.7922,0.798,0.7853,0.7978,0.7956,0.8071,0.7978,12.4578
239,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(52.233450742668325), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8067,0.8061,0.7917,0.7975,0.7853,0.7975,0.7952,0.8068,0.7975,16.047
240,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(100.0), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8067,0.805,0.7917,0.7975,0.7847,0.7971,0.7948,0.8066,0.7971,15.9061
